# Argumentation-01 — Le modèle de Toulmin (Rust / std)

[← Dung_AF_Semantics](Argumentation-03-Dung-AF-Semantics-Python.ipynb) | [↑ ArgumentAnalysis](README.md) | [Value_Based_AF →](Argumentation-03b-Value-Based-AF-Python.ipynb) | [ADF_Semantiques →](Argumentation-03d-ADF-Semantiques-Python.ipynb)

## Pourquoi ce notebook

Les notebooks [Argumentation-03-Dung-AF-Semantics-Python](Argumentation-03-Dung-AF-Semantics-Python.ipynb), [Argumentation-03b-Value-Based-AF-Python](Argumentation-03b-Value-Based-AF-Python.ipynb) et [Argumentation-03c-Ranking-Semantics-Python](Argumentation-03c-Ranking-Semantics-Python.ipynb) traitent l'argumentation **formelle/abstraite** : des arguments anonymes reliés par une relation d'attaque. Mais dans la pratique — un débat politique, un mémoire d'expertise, une plaidoirie — un argument n'est pas un nœud anonyme : il a une **structure interne**. Le philosophe **Stephen Toulmin** (1958, *The Uses of Argument*) a proposé une anatomie de cette structure, devenu l'outil de référence en analyse informelle de l'argumentation (rhétorique, droit, éthique appliquée, communication scientifique).

Toulmin observe qu'un argument complet ne se résume pas à « voici des faits, donc voici une conclusion ». Entre les faits (*Data*) et la conclusion (*Claim*), il y a toujours une **inférence autorisée** — le *Warrant* — qui dit *pourquoi* ces faits autorisent cette conclusion. Et ce warrant repose lui-même sur un *Backing* (un fondement), tolère un *Qualifier* (une modalité), et affronte des *Rebuttals* (des exceptions). Omettre ces pièces, c'est produire un argument qui *paraît* solide mais dont la chaîne logique est invisible — donc impossible à contester précisément.

Ce notebook reconstruit le **schéma à 6 composants** de Toulmin en Rust pur (`std` uniquement, kernel evcxr — le jumeau de la [version Python](Argumentation-01-Toulmin-Model-Python.ipynb) dont il suit le même fil), montre comment **auditer la complétude** d'un argument, et — c'est le pont computationnel — comment **traduire un débat d'arguments Toulmin en un cadre de Dung** : les rebuttals deviennent des attaques, et les sémantiques (grounded, preferred) décident quels claims survivent.

**Série Argumentation** | [Version Python](Argumentation-01-Toulmin-Model-Python.ipynb) | [Index](README.md) | Suivant : [01b-Schemes-Walton-Python](Argumentation-01b-Schemes-Walton-Python.ipynb)

## Le schéma à 6 composants

Un argument, selon Toulmin, se déploie en six pièces. Les trois premières sont **essentielles** (sans elles, il n'y a pas d'argument) ; les trois dernières le **qualifient** et le **durcissent**.

| Composant | Rôle | Question qu'il répond |
|-----------|------|------------------------|
| **Claim** (conclusion, *C*) | La thèse défendue | « Que veux-tu établir ? » |
| **Data** (données, *D*) | Les faits/observations invoqués | « Sur quoi t'appuies-tu ? » |
| **Warrant** (garant, *W*) | La règle d'inférence *D → C* | « Comment passes-tu de D à C ? » |
| **Backing** (fondement, *B*) | Le support du warrant (loi, autorité, théorie) | « Pourquoi cette règle vaut-elle ? » |
| **Qualifier** (modalité, *Q*) | La force de la conclusion (« nécessairement », « probablement ») | « Avec quelle certitude ? » |
| **Rebuttal** (réfutation, *R*) | Les conditions d'exception | « Quand ton argument tombe-t-il ? » |

La pièce **sous-enseignée** est le *Warrant*. Beaucoup d'arguments implicites sautent de *Data* à *Claim* sans expliciter la règle — et c'est précisément là que se logent les sophismes et les désaccords. Rendre le warrant explicite est l'acte analytique central du modèle Toulmin.

## Exemple concret : un argument complet

Pour rendre les composants concrets, déployons un argument familier :

> « *Le projet X doit probablement être financé, car son ROI projeté est de 15 % ; or les projets à ROI élevé méritent financement — principe d'allocation rationnelle du capital — sauf si la situation de défaut attendue le disqualifie. * »

Décomposé selon Toulmin :

- **Claim** : « Le projet X doit être financé »
- **Data** : « X a un ROI projeté de 15 % »
- **Warrant** : « Les projets à ROI élevé méritent financement »
- **Backing** : « Principe d'allocation rationnelle du capital »
- **Qualifier** : « probablement »
- **Rebuttal** (condition d'exception) : « sauf si le risque de défaut est prohibitif »

Implémentons ce schéma comme une structure Rust, puis vérifions sa complétude.

In [2]:
// --- Le schema de Toulmin (1958), pure std ---
use std::collections::BTreeSet;

/// Un argument structure selon le schema de Toulmin (6 composants).
///
/// Les 3 composants essentiels (claim, data, warrant) doivent etre non vides
/// pour que l'argument soit COMPLETE. Les 3 autres (backing, qualifier,
/// rebuttal_targets) le qualifient.
struct ToulminArgument {
    name: &'static str,
    claim: &'static str,
    data: &'static str,
    warrant: &'static str,
    backing: &'static str,
    qualifier: &'static str,
    rebuttal_targets: BTreeSet<&'static str>,
}

impl ToulminArgument {
    fn new(name: &'static str, claim: &'static str, data: &'static str, warrant: &'static str,
           backing: &'static str, qualifier: &'static str,
           rebuttal_targets: &[&'static str]) -> Self {
        ToulminArgument {
            name,
            claim,
            data,
            warrant,
            backing,
            qualifier,
            rebuttal_targets: rebuttal_targets.iter().copied().collect(),
        }
    }

    /// True si les 3 composants essentiels sont presents (claim/data/warrant).
    fn is_complete(&self) -> bool {
        !self.claim.is_empty() && !self.data.is_empty() && !self.warrant.is_empty()
    }

    /// Liste des composants essentiels manquants (pour diagnostic).
    fn missing_pieces(&self) -> Vec<&'static str> {
        let mut missing = Vec::new();
        if self.claim.is_empty() {
            missing.push("claim");
        }
        if self.data.is_empty() {
            missing.push("data");
        }
        if self.warrant.is_empty() {
            missing.push("warrant");
        }
        missing
    }

    /// Affiche l'argument deploye en ses 6 pieces.
    fn describe(&self) -> String {
        let rt: Vec<&str> = if self.rebuttal_targets.is_empty() {
            vec!["(aucun)"]
        } else {
            self.rebuttal_targets.iter().copied().collect()
        };
        [
            format!("[{}] Argument Toulmin (complet: {})", self.name, py_bool(self.is_complete())),
            format!("  Claim    : {}", or_fallback(self.claim, "(manquant)")),
            format!("  Data     : {}", or_fallback(self.data, "(manquant)")),
            format!("  Warrant  : {}", or_fallback(self.warrant, "(manquant)")),
            format!("  Backing  : {}", or_fallback(self.backing, "(non precise)")),
            format!("  Qualifier: {}", or_fallback(self.qualifier, "(non precise)")),
            format!("  Rebuttals cibles: {}", fmt_str_list(&rt)),
        ]
        .join("\n")
    }
}

/// Equivalent Rust de l'affichage Python d'un booleen (True/False).
fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

/// Liste Python ['a', 'b'] : apostrophes simples, virgule-espace, crochets.
fn fmt_str_list(items: &[&str]) -> String {
    let inner: Vec<String> = items.iter().map(|s| format!("'{}'", s)).collect();
    format!("[{}]", inner.join(", "))
}

/// Idiome Python `x or '(manquant)'` : chaine vide -> fallback.
fn or_fallback<'a>(s: &'a str, fb: &'a str) -> &'a str {
    if s.is_empty() { fb } else { s }
}

// Deploiement de l'exemple "projet X".
let arg_x = ToulminArgument::new(
    "X",
    "Le projet X doit etre finance",
    "X a un ROI projete de 15%",
    "Les projets a ROI eleve meritent financement",
    "Principe d'allocation rationnelle du capital",
    "probablement",
    &[], // on ajoutera la cible plus tard, avec l'argument adverse
);
println!("{}", arg_x.describe());
println!();
println!(
    "Complet ? {} ; pieces manquantes : {}",
    py_bool(arg_x.is_complete()),
    fmt_str_list(&arg_x.missing_pieces())
);

[X] Argument Toulmin (complet: True)


  Claim    : Le projet X doit etre finance


  Data     : X a un ROI projete de 15%


  Warrant  : Les projets a ROI eleve meritent financement


  Backing  : Principe d'allocation rationnelle du capital


  Qualifier: probablement


  Rebuttals cibles: ['(aucun)']


Complet ? True ; pieces manquantes : []


## Auditer la complétude

Le piège classique de l'argumentation implicite : un énoncé qui *ressemble* à un argument mais saute le warrant. « *X a un ROI de 15 %, donc X doit être financé* » omet la règle d'inférence — le lecteur doit la deviner. La méthode `missing_pieces` détecte ces trous, ce qui est précisément le travail d'un auditeur d'argumentation.

In [3]:
// Un argument INCOMPLET (warrant implicite) : diagnostic automatique.
use std::collections::BTreeSet;

// Chaque cellule est autonome sous evcxr : on redefinit le schema.
struct ToulminArgument {
    name: &'static str,
    claim: &'static str,
    data: &'static str,
    warrant: &'static str,
    backing: &'static str,
    qualifier: &'static str,
    rebuttal_targets: BTreeSet<&'static str>,
}

impl ToulminArgument {
    fn new(name: &'static str, claim: &'static str, data: &'static str, warrant: &'static str,
           backing: &'static str, qualifier: &'static str,
           rebuttal_targets: &[&'static str]) -> Self {
        ToulminArgument {
            name,
            claim,
            data,
            warrant,
            backing,
            qualifier,
            rebuttal_targets: rebuttal_targets.iter().copied().collect(),
        }
    }

    /// True si les 3 composants essentiels sont presents (claim/data/warrant).
    fn is_complete(&self) -> bool {
        !self.claim.is_empty() && !self.data.is_empty() && !self.warrant.is_empty()
    }

    /// Liste des composants essentiels manquants (pour diagnostic).
    fn missing_pieces(&self) -> Vec<&'static str> {
        let mut missing = Vec::new();
        if self.claim.is_empty() {
            missing.push("claim");
        }
        if self.data.is_empty() {
            missing.push("data");
        }
        if self.warrant.is_empty() {
            missing.push("warrant");
        }
        missing
    }

    /// Affiche l'argument deploye en ses 6 pieces.
    fn describe(&self) -> String {
        let rt: Vec<&str> = if self.rebuttal_targets.is_empty() {
            vec!["(aucun)"]
        } else {
            self.rebuttal_targets.iter().copied().collect()
        };
        [
            format!("[{}] Argument Toulmin (complet: {})", self.name, py_bool(self.is_complete())),
            format!("  Claim    : {}", or_fallback(self.claim, "(manquant)")),
            format!("  Data     : {}", or_fallback(self.data, "(manquant)")),
            format!("  Warrant  : {}", or_fallback(self.warrant, "(manquant)")),
            format!("  Backing  : {}", or_fallback(self.backing, "(non precise)")),
            format!("  Qualifier: {}", or_fallback(self.qualifier, "(non precise)")),
            format!("  Rebuttals cibles: {}", fmt_str_list(&rt)),
        ]
        .join("\n")
    }
}

/// Equivalent Rust de l'affichage Python d'un booleen (True/False).
fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

/// Liste Python ['a', 'b'] : apostrophes simples, virgule-espace, crochets.
fn fmt_str_list(items: &[&str]) -> String {
    let inner: Vec<String> = items.iter().map(|s| format!("'{}'", s)).collect();
    format!("[{}]", inner.join(", "))
}

/// Idiome Python `x or '(manquant)'` : chaine vide -> fallback.
fn or_fallback<'a>(s: &'a str, fb: &'a str) -> &'a str {
    if s.is_empty() { fb } else { s }
}

// Warrant implicite -> trou analytique. Les defauts Python (backing="",
// qualifier="", rebuttal_targets=()) deviennent des champs vides explicites.
let arg_leak = ToulminArgument::new(
    "leak",
    "X doit etre finance",
    "X a un ROI de 15%",
    "", // le warrant est implicite -> trou analytique
    "",
    "",
    &[],
);
println!("{}", arg_leak.describe());
println!();
let miss = arg_leak.missing_pieces();
if !miss.is_empty() {
    println!(
        ">>> Argument INCOMPLET : piece(s) essentielle(s) manquante(s) = {}",
        fmt_str_list(&miss)
    );
    println!(">>> Le warrant (D -> C) doit etre explicite : sur quelle regle d'inference passe-t-on du ROI a l'obligation de financer ?");
} else {
    println!(">>> Argument complet.");
};

[leak] Argument Toulmin (complet: False)


  Claim    : X doit etre finance


  Data     : X a un ROI de 15%


  Warrant  : (manquant)


  Backing  : (non precise)


  Qualifier: (non precise)


  Rebuttals cibles: ['(aucun)']


>>> Argument INCOMPLET : piece(s) essentielle(s) manquante(s) = ['warrant']


>>> Le warrant (D -> C) doit etre explicite : sur quelle regle d'inference passe-t-on du ROI a l'obligation de financer ?


## Le pont vers Dung : rebuttals = attaques

Voici le saut computationnel. Le modèle Toulmin décrit **un** argument isolé. Mais un débat réel en **confronte plusieurs** — et ces arguments se *réfutent* mutuellement (le composant Rebuttal). Or « *A réfute B* » est exactement « *A attaque B* » au sens de [Argumentation-03-Dung-AF-Semantics-Python](Argumentation-03-Dung-AF-Semantics-Python.ipynb). Donc :

1. On déploie chaque argument en sa forme Toulmin (6 composants).
2. On lit les `rebuttal_targets` de chacun → on construit la relation d'attaque du cadre de Dung.
3. On applique la sémantique *grounded* : les claims qui survivent sont ceux défendus dans le débat.

Construisons un débat à deux arguments qui se réfutent mutuellement, puis regardons le verdict du grounded.

In [4]:
// Primitives de Dung (comme Dung_AF_Semantics / Value_Based_AF).
use std::collections::BTreeSet;

struct ToulminArgument {
    name: &'static str,
    claim: &'static str,
    data: &'static str,
    warrant: &'static str,
    backing: &'static str,
    qualifier: &'static str,
    rebuttal_targets: BTreeSet<&'static str>,
}

impl ToulminArgument {
    fn new(name: &'static str, claim: &'static str, data: &'static str, warrant: &'static str,
           backing: &'static str, qualifier: &'static str,
           rebuttal_targets: &[&'static str]) -> Self {
        ToulminArgument {
            name,
            claim,
            data,
            warrant,
            backing,
            qualifier,
            rebuttal_targets: rebuttal_targets.iter().copied().collect(),
        }
    }

    /// True si les 3 composants essentiels sont presents (claim/data/warrant).
    fn is_complete(&self) -> bool {
        !self.claim.is_empty() && !self.data.is_empty() && !self.warrant.is_empty()
    }

    /// Liste des composants essentiels manquants (pour diagnostic).
    fn missing_pieces(&self) -> Vec<&'static str> {
        let mut missing = Vec::new();
        if self.claim.is_empty() {
            missing.push("claim");
        }
        if self.data.is_empty() {
            missing.push("data");
        }
        if self.warrant.is_empty() {
            missing.push("warrant");
        }
        missing
    }

    /// Affiche l'argument deploye en ses 6 pieces.
    fn describe(&self) -> String {
        let rt: Vec<&str> = if self.rebuttal_targets.is_empty() {
            vec!["(aucun)"]
        } else {
            self.rebuttal_targets.iter().copied().collect()
        };
        [
            format!("[{}] Argument Toulmin (complet: {})", self.name, py_bool(self.is_complete())),
            format!("  Claim    : {}", or_fallback(self.claim, "(manquant)")),
            format!("  Data     : {}", or_fallback(self.data, "(manquant)")),
            format!("  Warrant  : {}", or_fallback(self.warrant, "(manquant)")),
            format!("  Backing  : {}", or_fallback(self.backing, "(non precise)")),
            format!("  Qualifier: {}", or_fallback(self.qualifier, "(non precise)")),
            format!("  Rebuttals cibles: {}", fmt_str_list(&rt)),
        ]
        .join("\n")
    }
}

/// Equivalent Rust de l'affichage Python d'un booleen (True/False).
fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

/// Liste Python ['a', 'b'] : apostrophes simples, virgule-espace, crochets.
fn fmt_str_list(items: &[&str]) -> String {
    let inner: Vec<String> = items.iter().map(|s| format!("'{}'", s)).collect();
    format!("[{}]", inner.join(", "))
}

/// Idiome Python `x or '(manquant)'` : chaine vide -> fallback.
fn or_fallback<'a>(s: &'a str, fb: &'a str) -> &'a str {
    if s.is_empty() { fb } else { s }
}

/// Cadre d'argumentation abstrait de Dung (arguments + relation d'attaque).
struct Af {
    args: BTreeSet<&'static str>,
    attacks: BTreeSet<(&'static str, &'static str)>,
}

impl Af {
    fn attackers(&self, x: &str) -> BTreeSet<&'static str> {
        self.attacks
            .iter()
            .filter(|(_, b)| *b == x)
            .map(|(a, _)| *a)
            .collect()
    }
}

/// S defait x si un membre de S attaque x.
fn defeats(af: &Af, s: &BTreeSet<&str>, x: &str) -> bool {
    af.attacks.iter().any(|(a, b)| *b == x && s.contains(a))
}

/// S defend x si S defait chaque attaquant de x.
fn defends(af: &Af, s: &BTreeSet<&str>, x: &str) -> bool {
    af.attackers(x).iter().all(|b| defeats(af, s, b))
}

/// Extension grounded : point fixe E -> E union {x defendu par E}.
fn grounded(af: &Af) -> BTreeSet<&'static str> {
    let mut e: BTreeSet<&'static str> = BTreeSet::new();
    loop {
        let mut e_new = e.clone();
        for x in &af.args {
            if defends(af, &e, x) {
                e_new.insert(x);
            }
        }
        if e_new == e {
            return e;
        }
        e = e_new;
    }
}

/// Traduit un debat d'arguments Toulmin en un cadre de Dung :
/// chaque rebuttal_target (A refute B) devient une attaque (A -> B).
fn toulmin_debate_to_af(arguments: &[&ToulminArgument]) -> Af {
    let names: BTreeSet<&'static str> = arguments.iter().map(|a| a.name).collect();
    let mut attacks: BTreeSet<(&'static str, &'static str)> = BTreeSet::new();
    for a in arguments {
        for tgt in &a.rebuttal_targets {
            if names.contains(tgt) {
                // ne cree l'attaque que si la cible existe
                attacks.insert((a.name, *tgt));
            }
        }
    }
    Af { args: names, attacks }
}

/// Liste Python de tuples : [('A', 'B'), ('B', 'A')] — BTreeSet itere trie,
/// exactement le sorted() lexicographique Python sur tuples.
fn fmt_attack_list(attacks: &BTreeSet<(&'static str, &'static str)>) -> String {
    let inner: Vec<String> = attacks
        .iter()
        .map(|(a, b)| format!("('{}', '{}')", a, b))
        .collect();
    format!("[{}]", inner.join(", "))
}

// Un debat a deux arguments qui se refutent mutuellement (pro/contra).
let a = ToulminArgument::new(
    "A",
    "Le projet X doit etre finance",
    "X a un ROI projete de 15%",
    "Les projets a ROI eleve meritent financement",
    "Principe d'allocation rationnelle du capital",
    "probablement",
    &["B"], // A refute B
);
let b = ToulminArgument::new(
    "B",
    "Le projet X ne doit pas etre finance",
    "X presente un risque de defaut de 40%",
    "Les projets trop risques doivent etre evites",
    "Principe de prudence",
    "possiblement",
    &["A"], // B refute A
);

println!("=== Debat a 2 arguments (rebuttal mutuel) ===");
for arg in [&a, &b] {
    println!("{}", arg.describe());
    println!();
}

let af2 = toulmin_debate_to_af(&[&a, &b]);
let args_v: Vec<&str> = af2.args.iter().copied().collect();
println!(
    "Cadre de Dung : args={}, attacks={}",
    fmt_str_list(&args_v),
    fmt_attack_list(&af2.attacks)
);
let g2 = grounded(&af2);
let ext2 = if g2.is_empty() {
    "{} (vide)".to_string()
} else {
    fmt_str_list(&g2.iter().copied().collect::<Vec<&str>>())
};
println!("Extension grounded : {}", ext2);
println!(">>> Verdict : A<->B forment un cycle. Ni l'un ni l'autre ne defend l'autre");
println!("    -> grounded VIDE = standoff, aucune conclusion ne survit seule.");

=== Debat a 2 arguments (rebuttal mutuel) ===


[A] Argument Toulmin (complet: True)


  Claim    : Le projet X doit etre finance


  Data     : X a un ROI projete de 15%


  Warrant  : Les projets a ROI eleve meritent financement


  Backing  : Principe d'allocation rationnelle du capital


... (9 outputs left out)

Cadre de Dung : args=['A', 'B'], attacks=[('A', 'B'), ('B', 'A')]


Extension grounded : {} (vide)


>>> Verdict : A<->B forment un cycle. Ni l'un ni l'autre ne defend l'autre


    -> grounded VIDE = standoff, aucune conclusion ne survit seule.


## Le verdict bascule : introduire un arbitre

Le standoff (grounded vide) est insatisfaisant : le débat est *indécis*. Mais ajoutons un **troisième argument** *C* qui réfute *B* seulement (sans toucher à *A*). Concrètement, *C* conteste le *Data* de *B* (le risque de défaut est sur-estimé). Le cycle se brise : *C* défait *B*, donc *B* n'attaque plus *A*, donc *A* redevient défendable.

*Note de portage* : sous evcxr, rien ne persiste d'une cellule à l'autre — la cellule suivante redéfinit donc *A*, *B* et toutes les primitives de Dung avant d'ajouter *C*.

In [5]:
// On ajoute un 3e argument C qui refute B seulement (arbitre).
// Sous evcxr rien ne persiste : cette cellule redefinit A, B (litteraux
// identiques a la cellule precedente) et tout l'arsenal de Dung.
use std::collections::BTreeSet;

struct ToulminArgument {
    name: &'static str,
    claim: &'static str,
    data: &'static str,
    warrant: &'static str,
    backing: &'static str,
    qualifier: &'static str,
    rebuttal_targets: BTreeSet<&'static str>,
}

impl ToulminArgument {
    fn new(name: &'static str, claim: &'static str, data: &'static str, warrant: &'static str,
           backing: &'static str, qualifier: &'static str,
           rebuttal_targets: &[&'static str]) -> Self {
        ToulminArgument {
            name,
            claim,
            data,
            warrant,
            backing,
            qualifier,
            rebuttal_targets: rebuttal_targets.iter().copied().collect(),
        }
    }

    /// True si les 3 composants essentiels sont presents (claim/data/warrant).
    fn is_complete(&self) -> bool {
        !self.claim.is_empty() && !self.data.is_empty() && !self.warrant.is_empty()
    }

    /// Liste des composants essentiels manquants (pour diagnostic).
    fn missing_pieces(&self) -> Vec<&'static str> {
        let mut missing = Vec::new();
        if self.claim.is_empty() {
            missing.push("claim");
        }
        if self.data.is_empty() {
            missing.push("data");
        }
        if self.warrant.is_empty() {
            missing.push("warrant");
        }
        missing
    }

    /// Affiche l'argument deploye en ses 6 pieces.
    fn describe(&self) -> String {
        let rt: Vec<&str> = if self.rebuttal_targets.is_empty() {
            vec!["(aucun)"]
        } else {
            self.rebuttal_targets.iter().copied().collect()
        };
        [
            format!("[{}] Argument Toulmin (complet: {})", self.name, py_bool(self.is_complete())),
            format!("  Claim    : {}", or_fallback(self.claim, "(manquant)")),
            format!("  Data     : {}", or_fallback(self.data, "(manquant)")),
            format!("  Warrant  : {}", or_fallback(self.warrant, "(manquant)")),
            format!("  Backing  : {}", or_fallback(self.backing, "(non precise)")),
            format!("  Qualifier: {}", or_fallback(self.qualifier, "(non precise)")),
            format!("  Rebuttals cibles: {}", fmt_str_list(&rt)),
        ]
        .join("\n")
    }
}

/// Equivalent Rust de l'affichage Python d'un booleen (True/False).
fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

/// Liste Python ['a', 'b'] : apostrophes simples, virgule-espace, crochets.
fn fmt_str_list(items: &[&str]) -> String {
    let inner: Vec<String> = items.iter().map(|s| format!("'{}'", s)).collect();
    format!("[{}]", inner.join(", "))
}

/// Idiome Python `x or '(manquant)'` : chaine vide -> fallback.
fn or_fallback<'a>(s: &'a str, fb: &'a str) -> &'a str {
    if s.is_empty() { fb } else { s }
}

/// Cadre d'argumentation abstrait de Dung (arguments + relation d'attaque).
struct Af {
    args: BTreeSet<&'static str>,
    attacks: BTreeSet<(&'static str, &'static str)>,
}

impl Af {
    fn attackers(&self, x: &str) -> BTreeSet<&'static str> {
        self.attacks
            .iter()
            .filter(|(_, b)| *b == x)
            .map(|(a, _)| *a)
            .collect()
    }
}

/// S defait x si un membre de S attaque x.
fn defeats(af: &Af, s: &BTreeSet<&str>, x: &str) -> bool {
    af.attacks.iter().any(|(a, b)| *b == x && s.contains(a))
}

/// S defend x si S defait chaque attaquant de x.
fn defends(af: &Af, s: &BTreeSet<&str>, x: &str) -> bool {
    af.attackers(x).iter().all(|b| defeats(af, s, b))
}

/// Extension grounded : point fixe E -> E union {x defendu par E}.
fn grounded(af: &Af) -> BTreeSet<&'static str> {
    let mut e: BTreeSet<&'static str> = BTreeSet::new();
    loop {
        let mut e_new = e.clone();
        for x in &af.args {
            if defends(af, &e, x) {
                e_new.insert(x);
            }
        }
        if e_new == e {
            return e;
        }
        e = e_new;
    }
}

/// Traduit un debat d'arguments Toulmin en un cadre de Dung :
/// chaque rebuttal_target (A refute B) devient une attaque (A -> B).
fn toulmin_debate_to_af(arguments: &[&ToulminArgument]) -> Af {
    let names: BTreeSet<&'static str> = arguments.iter().map(|a| a.name).collect();
    let mut attacks: BTreeSet<(&'static str, &'static str)> = BTreeSet::new();
    for a in arguments {
        for tgt in &a.rebuttal_targets {
            if names.contains(tgt) {
                // ne cree l'attaque que si la cible existe
                attacks.insert((a.name, *tgt));
            }
        }
    }
    Af { args: names, attacks }
}

/// Liste Python de tuples : [('A', 'B'), ('B', 'A')] — BTreeSet itere trie,
/// exactement le sorted() lexicographique Python sur tuples.
fn fmt_attack_list(attacks: &BTreeSet<(&'static str, &'static str)>) -> String {
    let inner: Vec<String> = attacks
        .iter()
        .map(|(a, b)| format!("('{}', '{}')", a, b))
        .collect();
    format!("[{}]", inner.join(", "))
}

let a = ToulminArgument::new(
    "A",
    "Le projet X doit etre finance",
    "X a un ROI projete de 15%",
    "Les projets a ROI eleve meritent financement",
    "Principe d'allocation rationnelle du capital",
    "probablement",
    &["B"], // A refute B
);
let b = ToulminArgument::new(
    "B",
    "Le projet X ne doit pas etre finance",
    "X presente un risque de defaut de 40%",
    "Les projets trop risques doivent etre evites",
    "Principe de prudence",
    "possiblement",
    &["A"], // B refute A
);
let c = ToulminArgument::new(
    "C",
    "Le risque de defaut de X est surestime",
    "L'historique montre 10% de defaut reel sur projets similaires",
    "Les projections de risque doivent etre calibrees sur donnees passees",
    "Principe de calibration empirique",
    "fortement",
    &["B"], // C refute B, MAIS PAS A
);

println!("=== Debat a 3 arguments (C arbitre en refutant B) ===");
println!("{}", c.describe());
println!();

let af3 = toulmin_debate_to_af(&[&a, &b, &c]);
let args3: Vec<&str> = af3.args.iter().copied().collect();
println!("Cadre de Dung : args={}", fmt_str_list(&args3));
println!("                attacks={}", fmt_attack_list(&af3.attacks));
let g3 = grounded(&af3);
println!("Extension grounded : {}", fmt_str_list(&g3.iter().copied().collect::<Vec<&str>>()));
println!();
println!(">>> Lecture : C defait B -> B ne peut plus attaquer A -> A est defendu.");
println!("    Les claims A et C survivent ; B (le contre-argument) est exclu.");
println!("    Le warrant sous-jacent : un seul arbitre peut faire basculer un debat");
println!("    indécis vers une conclusion, SANS toucher aux arguments principaux.");

=== Debat a 3 arguments (C arbitre en refutant B) ===


[C] Argument Toulmin (complet: True)


  Claim    : Le risque de defaut de X est surestime


  Data     : L'historique montre 10% de defaut reel sur projets similaires


  Warrant  : Les projections de risque doivent etre calibrees sur donnees passees


  Backing  : Principe de calibration empirique


... (5 outputs left out)

>>> Lecture : C defait B -> B ne peut plus attaquer A -> A est defendu.


    Les claims A et C survivent ; B (le contre-argument) est exclu.


    Le warrant sous-jacent : un seul arbitre peut faire basculer un debat


    indécis vers une conclusion, SANS toucher aux arguments principaux.


## Vérifiez votre compréhension

Les trois exercices approfondissent le modèle. Chaque stub est à compléter — le notebook s'exécute de bout en bout même non complété (les exercices renvoient `None`/valeurs neutres et affichent un message). La méthode de chaque exercice est déjà démontrée sur un exemple dans le carnet :

- **Exercice 1** (diagnostiquer un argument incomplet) → la section *Auditer la complétude* l'a fait sur l'argument « leak » : `Complet ? False ; pieces manquantes : ['warrant']`.
- **Exercice 2** (construire un standoff) → la section *Le pont vers Dung* l'a fait avec A et B : `Extension grounded : {} (vide)`.
- **Exercice 3** (trouver l'arbitre qui bascule le débat) → la section *Le verdict bascule* l'a fait avec C : `Extension grounded : ['A', 'C']`.

### Exercice 1 — Diagnostiquer un argument incomplet

Étant donné un énoncé familier d'opinion, identifiez la pièce essentielle manquante et complétez l'argument Toulmin pour qu'il soit `is_complete() == True`.

**Objectif** : prendre l'énoncé « *Il pleut, donc je prends mon parapluie* », le modéliser (claim + data fournis, warrant à expliciter), et le rendre complet.

In [6]:
// Exercice 1 : a completer
use std::collections::BTreeSet;

// Le type figure dans la signature du retour : chaque cellule etant
// autonome sous evcxr, on redefinit le schema de Toulmin.
struct ToulminArgument {
    name: &'static str,
    claim: &'static str,
    data: &'static str,
    warrant: &'static str,
    backing: &'static str,
    qualifier: &'static str,
    rebuttal_targets: BTreeSet<&'static str>,
}

/// Renvoie un ToulminArgument complet modelisant : 'Il pleut, donc je
/// prends mon parapluie'.
///
/// Indice : le DATA = 'il pleut', le CLAIM = 'je prends mon parapluie'.
/// Etape 1 : quel est le WARRANT (regle D -> C) implicite ? L'expliciter.
/// Etape 2 : construire le ToulminArgument avec les 3 composants essentiels.
fn argument_parapluie_complet() -> Option<ToulminArgument> {
    // TODO etudiant
    None
}

// let arg = argument_parapluie_complet();
// if let Some(a) = arg {
//     println!("{}", a.describe());
//     let miss = a.missing_pieces();
//     if !miss.is_empty() {
//         println!("ENCORE INCOMPLET : manque {}", fmt_str_list(&miss));
//     } else {
//         println!("SUCCES : argument complet (warrant explicite).");
//     }
// } else {
//     println!("Exercice a completer.");
// }
println!("Exercice 1 a completer.");

Exercice 1 a completer.


### Exercice 2 — Construire un débat équilibré (standoff)

Construisez **deux** arguments Toulmin complets qui se réfutent mutuellement (comme A et B ci-dessus), sur un sujet de votre choix (ex. « *le télétravail devrait être la norme* »). Vérifiez via `toulmin_debate_to_af` + `grounded` que le débat est un **standoff** (grounded vide).

**Objectif** : sentir que deux arguments *corrects* mais opposés produisent une indécision — c'est pourquoi un débat pur pro/contra est rarement tranché sans un troisième élément.

In [7]:
// Exercice 2 : a completer
use std::collections::BTreeSet;

// Autonomie evcxr : on redefinit le schema de Toulmin.
struct ToulminArgument {
    name: &'static str,
    claim: &'static str,
    data: &'static str,
    warrant: &'static str,
    backing: &'static str,
    qualifier: &'static str,
    rebuttal_targets: BTreeSet<&'static str>,
}

/// Construisez deux ToulminArgument complets (PRO et CONTRA le teletravail
/// comme norme) qui se refutent mutuellement. Renvoie (pro, contra).
///
/// Indice : chaque argument doit avoir claim + data + warrant non vides, et
/// rebuttal_targets pointant vers le nom de l'autre.
fn debat_teletravail_standoff() -> (Option<ToulminArgument>, Option<ToulminArgument>) {
    // TODO etudiant
    (None, None)
}

// let (pro, contra) = debat_teletravail_standoff();
// if let (Some(p), Some(c)) = (pro, contra) {
//     let af_ex2 = toulmin_debate_to_af(&[&p, &c]);
//     let g_ex2 = grounded(&af_ex2);
//     println!("Attaques : {}", fmt_attack_list(&af_ex2.attacks));
//     let g = if g_ex2.is_empty() {
//         "{} (vide = standoff)".to_string()
//     } else {
//         fmt_str_list(&g_ex2.iter().copied().collect::<Vec<&str>>())
//     };
//     println!("Grounded : {}", g);
//     if g_ex2.is_empty() {
//         println!("SUCCES : standoff confirme (debat equilibre).");
//     }
// } else {
//     println!("Exercice a completer.");
// }
println!("Exercice 2 a completer.");

Exercice 2 a completer.


### Exercice 3 — Trouver l'arbitre qui fait basculer le débat

Reprenant votre débat de l'exercice 2 (pro/contra en standoff), construisez un **troisième** argument *C* qui réfute *un seul* des deux, de sorte que le grounded passe de vide à une conclusion non vide. Identifiez lequel des deux claims survit.

**Objectif** : comprendre que la conclusion d'un débat Toulmin-Dung dépend de *où* l'arbitre frappe — réfuter *pro* fait gagner *contra*, et inversement.

In [8]:
// Exercice 3 : a completer
use std::collections::BTreeSet;

// Autonomie evcxr : on redefinit le schema de Toulmin.
struct ToulminArgument {
    name: &'static str,
    claim: &'static str,
    data: &'static str,
    warrant: &'static str,
    backing: &'static str,
    qualifier: &'static str,
    rebuttal_targets: BTreeSet<&'static str>,
}

/// Construit un 3e argument C qui refute `cible` (parmi 'pro'/'contra')
/// et renvoie (C, ensemble_grounded_obtenu).
///
/// Indice : rebuttal_targets de C = {nom de l'argument cible}. Puis calculer
/// le grounded du debat a 3 arguments. Renvoyer (C, set(grounded)).
fn arbitre_qui_bascule(
    _pro: &ToulminArgument,
    _contra: &ToulminArgument,
    _cible: &str,
) -> (Option<ToulminArgument>, Option<BTreeSet<&'static str>>) {
    // TODO etudiant
    (None, None)
}

// Supposons pro.name="PRO", contra.name="CONTRA" (a adapter a votre exo 2).
// let (c_opt, g) = arbitre_qui_bascule(&pro, &contra, "CONTRA");
// if let Some(cc) = c_opt {
//     let g_v: Vec<&str> = g.unwrap().into_iter().collect();
//     println!("Arbitre C refute CONTRA -> grounded = {}", fmt_str_list(&g_v));
//     println!("Le claim survivant est celui dont l'adversaire a ete refute.");
// } else {
//     println!("Exercice a completer (depend de l'exercice 2).");
// }
println!("Exercice 3 a completer.");

Exercice 3 a completer.


## À retenir

- Un argument n'est pas un bloc monolithique : le **schéma de Toulmin** le déploie en six pièces, dont trois **essentielles** (claim, data, warrant). Le *warrant* — la règle d'inférence *D → C* — est la pièce la plus souvent laissée implicite : l'audit l'a pris sur le vif avec l'argument « leak », `Complet ? False ; piece(s) essentielle(s) manquante(s) = ['warrant']`, là où l'argument complet `[X]` rend `Complet ? True ; pieces manquantes : []`.
- Auditer la complétude (`is_complete` / `missing_pieces`) est un acte analytique : il force à expliciter la chaîne logique plutôt qu'à la présupposer — et le diagnostic imprimé dit *pourquoi* le warrant manque : « sur quelle règle d'inférence passe-t-on du ROI à l'obligation de financer ? »
- **Le pont vers Dung** : en déployant plusieurs arguments Toulmin et en lisant leurs rebuttals comme des attaques, on obtient un cadre de Dung (`args=['A', 'B'], attacks=[('A', 'B'), ('B', 'A')]`). Les sémantiques tranchent alors le débat : un standoff (`Extension grounded : {} (vide)`) signifie que le débat est *structurellement* indécis.
- Un seul arbitre suffit à faire basculer un débat indécis : en ajoutant C qui attaque B, l'extension grounded passe de `{}` à `['A', 'C']` — sans toucher aux arguments principaux. C'est la prémisse du warrant « un seul arbitre peut faire basculer un débat indécis » que le carnet a mesurée plutôt qu'affirmée.

## Pour aller plus loin

- **Jumeau Python** : [Argumentation-01-Toulmin-Model-Python](Argumentation-01-Toulmin-Model-Python.ipynb) — la version originale de ce carnet, dont le portage Rust suit le fil à l'identique (aucun jumeau C# n'existe pour cette série).
- [Index de la sous-série ArgumentAnalysis](README.md) — la place du modèle de Toulmin dans le parcours.
- [Index de la série SymbolicAI](../README.md) — le fil complet, du raisonnement formel aux systèmes multi-agents.

**Argumentation dialectique (ADF)** : [Argumentation-03d-ADF-Semantiques-Python](Argumentation-03d-ADF-Semantiques-Python.ipynb) remplace l'attaque binaire par une *condition d'acceptation* booléenne sur les parents ; le résultat d'une sémantique y devient une interprétation à valeurs `t` (accepté) / `f` (refusé) / `u` (indécis) — là où le `grounded` de Dung laisse un cadre entièrement vide, l'ADF laisse des arguments *indécis*.

**Argumentation graduée** : plutôt qu'un verdict accepté/rejeté, [Argumentation-03c-Ranking-Semantics-Python](Argumentation-03c-Ranking-Semantics-Python.ipynb) attribue une *force* numérique à chaque argument — utile quand un débat ne se résout pas par grounded seul.

**Valeurs et audiences** : les warrants reposent souvent sur des *valeurs* (cf. [Argumentation-03b-Value-Based-AF-Python](Argumentation-03b-Value-Based-AF-Python.ipynb)) ; deux audiences aux valeurs différentes peuvent accepter le même *Data* mais rejeter le *Warrant* — d'où l'intérêt de croiser Toulmin (structure) et VAF (préférences).

**Argumentation bipolaire** : Toulmin distingue l'attaque (rebuttal) du *support* implicite entre data et claim ; les cadres bipolaires (Cayrol & Lagasquie-Schiex, 2005) modélisent explicitement une relation de support en plus de l'attaque.

**Référence** — Stephen Toulmin, *The Uses of Argument* (Cambridge University Press, 1958, mis à jour 2003), le livre fondateur du schéma à six composants reconstruit ici.

***

*Notebook pédagogique — Rust `std` uniquement (kernel evcxr), déterministe, sans LLM ni solveur externe. Conforme C.1 (stubs sans erreur volontaire), C.2 (sorties réelles), exercices à compléter.*